# GR-RHS-friendly high-dimensional grouped simulation

这个 notebook 是在 `Simulation.ipynb` 思路基础上的改写版。

核心修改：

1. 保留原 notebook 中“高维、复杂相关、可有正负相关和 cross-loading”的 `X` 生成思想。
2. 修改 `beta` 生成方式：不再把非零系数随机撒到全体变量中，而是先选择 active groups，再在 active groups 内生成 concentrated / distributed / mixed 信号。
3. 新增 decoy groups：这些组可以和 active groups 有相关结构，但真实 beta 为 0，用来测试模型能否区分“相关但无效”的组。
4. 输出 group-level diagnostics，验证数据是否真的形成了 active/null group 结构。

为什么这样更适合 GR-RHS：GR-RHS 的优势不只是处理高维相关变量，而是同时利用 group-level shrinkage 和 within-group local escape。因此验证数据应该让 `X` 有组相关，同时让 `beta` 也有 active/null group 结构，并允许 active group 内部异质。

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    import seaborn as sns
except ModuleNotFoundError:
    # 如果当前环境没有 seaborn，后面的诊断图会自动退回到 matplotlib。
    sns = None


## 1. Correlated high-dimensional design matrix

这一部分基本保留 `Simulation.ipynb` 的精神：用 signed factor block model 生成复杂相关的高维特征矩阵。

和普通 block-diagonal 设计相比，这里允许：

- 组内正负相关混合；
- 组大小不完全相等；
- 组间 factor 相关；
- 部分变量 cross-loading 到其他组；
- 少量 orphan/noise variables。

这更接近真实 omics 数据里的局部相关 + 全局因子 + 交叉相关结构。

In [ ]:
def nearest_correlation(C: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    """Project a symmetric matrix to a numerically valid correlation matrix."""
    C = np.asarray(C, dtype=float)
    C = 0.5 * (C + C.T)
    w, V = np.linalg.eigh(C)
    w = np.maximum(w, eps)
    C_psd = (V * w) @ V.T
    d = np.sqrt(np.maximum(np.diag(C_psd), eps))
    C_corr = C_psd / d[:, None] / d[None, :]
    np.fill_diagonal(C_corr, 1.0)
    return C_corr


def sample_signed_factor_group_design(
    n: int,
    group_sizes: list[int],
    *,
    within_abs_rho: float = 0.65,
    neg_prob_within: float = 0.35,
    loading_mag_jitter: float = 0.25,
    between_factor_rho_mean: float = 0.10,
    between_factor_rho_sd: float = 0.08,
    cross_loading_frac: float = 0.10,
    cross_loading_scale: float = 0.25,
    orphan_frac: float = 0.00,
    random_state: int = 20260428,
) -> tuple[pd.DataFrame, list[np.ndarray], dict]:
    """Generate high-dimensional grouped X with realistic correlation structure.

    MODIFIED/KEPT FROM Simulation.ipynb:
    - 保留 signed factor block model 作为 X 的生成机制。
    - 改为显式输入 group_sizes，而不是只输入 n_clusters 后随机产生超大 cluster。
    - 原因：GR-RHS benchmark 需要明确的 groups，例如 50 groups x 10 features，
      这样后续 active/null group 结构才清楚，也更容易和论文里的 grouped prior 对齐。
    """
    rng = np.random.default_rng(random_state)
    group_sizes = [int(s) for s in group_sizes]
    if min(group_sizes) <= 0:
        raise ValueError("All group sizes must be positive.")

    G = len(group_sizes)
    p = int(sum(group_sizes))
    groups = []
    start = 0
    for size in group_sizes:
        idx = np.arange(start, start + size, dtype=int)
        groups.append(idx)
        start += size

    membership = np.empty(p, dtype=int)
    for g, idx in enumerate(groups):
        membership[idx] = g

    orphan_frac = float(np.clip(orphan_frac, 0.0, 1.0))
    n_orphan = int(round(p * orphan_frac))
    if n_orphan > 0:
        orphan_idx = rng.choice(p, size=n_orphan, replace=False)
        membership[orphan_idx] = -1

    L = np.zeros((p, G), dtype=float)
    for j in range(p):
        g = membership[j]
        if g < 0:
            continue
        sign = -1.0 if rng.random() < neg_prob_within else 1.0
        mag = max(0.05, rng.normal(1.0, loading_mag_jitter))
        L[j, g] = sign * mag

        if G > 1 and rng.random() < cross_loading_frac:
            other = rng.integers(0, G - 1)
            other = other + (other >= g)
            sign2 = -1.0 if rng.random() < 0.5 else 1.0
            L[j, other] = sign2 * mag * cross_loading_scale

    B = np.eye(G, dtype=float)
    if G > 1:
        iu = np.triu_indices(G, 1)
        vals = rng.normal(between_factor_rho_mean, between_factor_rho_sd, size=len(iu[0]))
        vals = np.clip(vals, -0.75, 0.75)
        B[iu] = vals
        B[(iu[1], iu[0])] = vals
    B = nearest_correlation(B)

    within_abs_rho = float(np.clip(within_abs_rho, 1e-6, 0.99))
    noise_var = (1.0 - within_abs_rho) / within_abs_rho
    noise_std = float(np.sqrt(max(noise_var, 1e-8)))

    cholB = np.linalg.cholesky(B)
    F = rng.standard_normal((int(n), G)) @ cholB.T
    E = rng.normal(0.0, noise_std, size=(int(n), p))
    X = F @ L.T + E

    # Scale by population diagonal implied by the factor model.
    LB = L @ B
    diag_S = np.sum(LB * L, axis=1) + noise_std**2
    X = X / np.sqrt(np.maximum(diag_S, 1e-12))[None, :]

    # Final sample standardization keeps downstream methods on the same scale.
    X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, ddof=0, keepdims=True) + 1e-12)

    columns = [f"x{j + 1}" for j in range(p)]
    meta = {
        "n": int(n),
        "p": int(p),
        "n_groups": int(G),
        "group_sizes": group_sizes,
        "membership": membership.tolist(),
        "within_abs_rho_target": float(within_abs_rho),
        "between_factor_rho_mean": float(between_factor_rho_mean),
        "between_factor_rho_sd": float(between_factor_rho_sd),
        "cross_loading_frac": float(cross_loading_frac),
        "cross_loading_scale": float(cross_loading_scale),
        "orphan_frac": float(orphan_frac),
    }
    return pd.DataFrame(X, columns=columns), groups, meta


## 2. GR-RHS-friendly grouped beta generator

这是相对原 `Simulation.ipynb` 最关键的修改。

原 notebook 的 `beta` 是从全体变量随机抽非零坐标，类似“把信号撒胡椒粉一样撒到所有 feature 上”。这样只能测试普通 high-dimensional sparse regression，不能真正测试 group-aware shrinkage。

这里改成：

1. 先选 active groups；
2. null groups 的 beta 全为 0；
3. active groups 内部允许三种信号形态：
   - `concentrated`: 少数强坐标；
   - `distributed`: 多个弱到中等坐标；
   - `mixed`: 一个主效应 + 若干弱效应；
4. 每个 active group 可以有不同形态，形成 multimode heterogeneous signals。

这更适合验证 GR-RHS，因为 GR-RHS 要证明的正是：组层保留 active group，同时局部尺度允许组内异质逃逸。

In [ ]:
def make_grrhs_grouped_beta(
    groups: list[np.ndarray],
    *,
    n_active_groups: int = 5,
    active_groups: list[int] | None = None,
    signal_modes: tuple[str, ...] = ("concentrated", "distributed", "mixed"),
    energy_per_active_group: str = "dirichlet",
    total_signal_energy: float = 1.0,
    support_fraction_range: tuple[float, float] = (0.2, 0.8),
    random_state: int = 20260428,
) -> tuple[pd.Series, dict]:
    """Generate beta with explicit active/null group structure.

    MODIFIED FROM Simulation.ipynb:
    - 原始版本：causal_idx/nonzero_idx 从所有 p 个变量随机抽。
    - 这里：先抽 active groups，再只在 active groups 内生成非零 beta。

    WHY THIS IS BETTER FOR GR-RHS:
    - GR-RHS 的 group-specific slab ceiling 需要真实的 active/null group 对比。
    - 组内 concentrated/distributed/mixed 的差异能测试 local scales 是否能处理异质信号。
    - multimode active groups 能测试一个统一 shrinkage 模型是否过度假设所有组形态相同。
    """
    rng = np.random.default_rng(random_state)
    G = len(groups)
    p = int(sum(len(g) for g in groups))
    if active_groups is None:
        active_groups = sorted(rng.choice(G, size=int(n_active_groups), replace=False).tolist())
    else:
        active_groups = sorted(int(g) for g in active_groups)
    if any(g < 0 or g >= G for g in active_groups):
        raise ValueError("active_groups contains an invalid group id.")

    K = len(active_groups)
    if K == 0:
        raise ValueError("At least one active group is required.")

    if energy_per_active_group == "equal":
        group_energy = np.full(K, 1.0 / K)
    elif energy_per_active_group == "dirichlet":
        group_energy = rng.dirichlet(np.full(K, 3.0))
    else:
        raise ValueError("energy_per_active_group must be 'equal' or 'dirichlet'.")
    group_energy = group_energy * float(total_signal_energy)

    beta = np.zeros(p, dtype=float)
    group_records = []
    mode_by_group = {}

    for k, g in enumerate(active_groups):
        idx = np.asarray(groups[g], dtype=int)
        p_g = len(idx)
        mode = str(rng.choice(signal_modes))
        mode_by_group[g] = mode

        if mode == "concentrated":
            # 少数强坐标：测试模型是否允许 active group 中的局部逃逸。
            s = max(1, int(round(0.15 * p_g)))
            active_idx = rng.choice(idx, size=s, replace=False)
            weights = rng.dirichlet(np.full(s, 0.35))
        elif mode == "distributed":
            # 多个弱/中等坐标：测试 group-level retention 是否能保护分散信号。
            lo, hi = support_fraction_range
            frac = float(rng.uniform(lo, hi))
            s = max(2, min(p_g, int(round(frac * p_g))))
            active_idx = rng.choice(idx, size=s, replace=False)
            weights = rng.dirichlet(np.full(s, 4.0))
        elif mode == "mixed":
            # 一个主效应 + 一批弱效应：这是很多 genomic block 更像的形态。
            s = max(2, min(p_g, int(round(0.5 * p_g))))
            active_idx = rng.choice(idx, size=s, replace=False)
            weights = np.full(s, 0.0)
            weights[0] = 0.55
            if s > 1:
                weights[1:] = 0.45 * rng.dirichlet(np.full(s - 1, 2.0))
        else:
            raise ValueError(f"Unknown signal mode: {mode}")

        sign = -1.0 if rng.random() < 0.5 else 1.0
        magnitudes = np.sqrt(group_energy[k] * weights)
        beta[active_idx] = sign * magnitudes

        group_records.append(
            {
                "group": int(g),
                "mode": mode,
                "group_size": int(p_g),
                "n_nonzero": int(len(active_idx)),
                "energy": float(group_energy[k]),
                "sign": int(sign),
                "nonzero_indices": active_idx.astype(int).tolist(),
            }
        )

    meta = {
        "active_groups": active_groups,
        "null_groups": [g for g in range(G) if g not in active_groups],
        "mode_by_group": mode_by_group,
        "group_records": group_records,
        "n_nonzero": int(np.sum(beta != 0.0)),
        "total_signal_energy": float(np.sum(beta**2)),
    }
    return pd.Series(beta, index=[f"x{j + 1}" for j in range(p)], name="beta"), meta


## 3. Full simulation wrapper

这里把复杂相关的 `X` 和 GR-RHS 友好的 `beta` 合并起来，并按目标 SNR 生成连续响应。

注意：这里默认使用 `signal_to_noise = sd(X beta) / sd(noise)`。如果要和 `Simulation_second` 完全统一，也可以改成 target R2 校准。

In [ ]:
def simulate_grrhs_highdim_grouped_data(
    *,
    n: int = 200,
    group_sizes: list[int] | None = None,
    n_active_groups: int = 5,
    signal_to_noise: float = 2.0,
    random_state: int = 20260428,
    **x_kwargs,
) -> dict:
    """Simulate a high-dimensional dataset designed for GR-RHS validation.

    MODIFIED DESIGN SUMMARY:
    - X: keeps realistic high-dimensional correlation from the original notebook.
    - beta: changed to active/null group signal instead of random coordinate sprinkling.
    - y: generated from X @ beta with calibrated noise.
    """
    rng = np.random.default_rng(random_state)
    if group_sizes is None:
        group_sizes = [10] * 50  # p = 500, p > n by default.

    X, groups, x_meta = sample_signed_factor_group_design(
        n=n,
        group_sizes=group_sizes,
        random_state=random_state,
        **x_kwargs,
    )
    beta, beta_meta = make_grrhs_grouped_beta(
        groups,
        n_active_groups=n_active_groups,
        random_state=random_state + 1009,
    )

    linpred = X.to_numpy() @ beta.to_numpy()
    noise_sd = float(np.std(linpred, ddof=0) / max(float(signal_to_noise), 1e-12))
    y = linpred + rng.normal(0.0, noise_sd, size=int(n))
    y = pd.Series(y, name="y")

    meta = {
        "design": x_meta,
        "signal": beta_meta,
        "signal_to_noise_target": float(signal_to_noise),
        "noise_sd": noise_sd,
        "empirical_signal_sd": float(np.std(linpred, ddof=0)),
        "empirical_noise_sd": noise_sd,
        "empirical_r2_approx": float(np.var(linpred) / (np.var(linpred) + noise_sd**2)),
    }

    return {"X": X, "y": y, "beta": beta, "groups": groups, "meta": meta}


## 4. Example: p = 500, n = 200 high-dimensional grouped data

这个例子是推荐作为 GR-RHS 高维 synthetic benchmark 的起点：

- `n = 200`
- `p = 500`
- `50` groups，每组 `10` 个变量
- `5` 个 active groups
- active group 内部信号形态随机为 concentrated / distributed / mixed

它比原 notebook 的 `n=50, p=3000, n_clusters=3` 更适合模型验证，因为组数量多、组大小清楚、active/null group 对比明确。

In [ ]:
sim = simulate_grrhs_highdim_grouped_data(
    n=200,
    group_sizes=[10] * 50,
    n_active_groups=5,
    signal_to_noise=2.0,
    within_abs_rho=0.70,
    neg_prob_within=0.35,
    between_factor_rho_mean=0.10,
    between_factor_rho_sd=0.08,
    cross_loading_frac=0.10,
    cross_loading_scale=0.25,
    random_state=20260428,
)

X = sim["X"]
y = sim["y"]
beta = sim["beta"]
groups = sim["groups"]
meta = sim["meta"]

print(json.dumps({
    "n": meta["design"]["n"],
    "p": meta["design"]["p"],
    "n_groups": meta["design"]["n_groups"],
    "active_groups": meta["signal"]["active_groups"],
    "n_nonzero_beta": meta["signal"]["n_nonzero"],
    "empirical_r2_approx": round(meta["empirical_r2_approx"], 3),
}, indent=2))


## 5. Diagnostics: does the simulation really create grouped signal?

下面这些诊断是为了避免一个常见问题：数据看起来高维、相关、复杂，但真实信号其实没有 group structure。

如果这个表显示 active groups 的 beta energy 明显大于 null groups，说明修改成功：这个数据集确实能验证 GR-RHS 的 group-aware shrinkage。

In [ ]:
def group_signal_summary(beta: pd.Series, groups: list[np.ndarray], active_groups: list[int]) -> pd.DataFrame:
    rows = []
    b = beta.to_numpy()
    active_set = set(active_groups)
    for g, idx in enumerate(groups):
        vals = b[idx]
        rows.append(
            {
                "group": g,
                "status": "active" if g in active_set else "null",
                "group_size": len(idx),
                "n_nonzero": int(np.sum(vals != 0.0)),
                "beta_l2": float(np.sqrt(np.sum(vals**2))),
                "beta_energy": float(np.sum(vals**2)),
                "max_abs_beta": float(np.max(np.abs(vals))),
            }
        )
    return pd.DataFrame(rows)


summary = group_signal_summary(beta, groups, meta["signal"]["active_groups"])
summary.sort_values(["status", "beta_energy"], ascending=[True, False]).head(12)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_df = summary.copy()
if sns is not None:
    sns.barplot(data=plot_df, x="group", y="beta_energy", hue="status", dodge=False, ax=axes[0])
else:
    colors = np.where(plot_df["status"].to_numpy() == "active", "tab:red", "tab:blue")
    axes[0].bar(plot_df["group"], plot_df["beta_energy"], color=colors)
axes[0].set_title("True beta energy by group")
axes[0].set_xlabel("Group")
axes[0].set_ylabel("sum(beta^2)")
axes[0].tick_params(axis="x", labelrotation=90, labelsize=6)

nz = beta.to_numpy() != 0
axes[1].stem(np.arange(len(beta)), beta.to_numpy(), markerfmt=".", basefmt=" ")
axes[1].set_title("True beta vector")
axes[1].set_xlabel("Feature index")
axes[1].set_ylabel("beta")

plt.tight_layout()
plt.show()


## 6. Diagnostics: does X have grouped correlation?

这里看 feature correlation heatmap。由于 `p=500`，完整 clustermap 可能稍慢；可以先抽取前 15 个组，也就是前 150 个变量。

In [ ]:
subset_p = 150
C = X.iloc[:, :subset_p].corr()
if sns is not None:
    sns.clustermap(C, cmap="coolwarm", center=0.0, figsize=(7, 7), xticklabels=False, yticklabels=False)
    plt.show()
else:
    plt.figure(figsize=(7, 6))
    plt.imshow(C, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    plt.colorbar(label="correlation")
    plt.title("Feature correlation heatmap")
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
    plt.show()


## 7. Optional export

如果要把这个数据接入你的 benchmark runner，可以导出 `X.csv`, `y.csv`, `beta.csv`, `groups.json`, `meta.json`。

默认先不执行导出，避免意外产生大文件。

In [ ]:
EXPORT = False

if EXPORT:
    out_dir = Path("outputs/history/grrhs_highdim_synthetic_notebook")
    out_dir.mkdir(parents=True, exist_ok=True)
    X.to_csv(out_dir / "X.csv", index=False)
    y.to_csv(out_dir / "y.csv", index=False)
    beta.to_csv(out_dir / "beta.csv")
    with (out_dir / "groups.json").open("w", encoding="utf-8") as f:
        json.dump([g.astype(int).tolist() for g in groups], f, indent=2)
    with (out_dir / "meta.json").open("w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved to {out_dir}")
